# REST API Fundamentals

**REST** (Representational State Transfer) is an architectural style for machine-to-machine communication over HTTP, defined by Roy Fielding in his 2000 doctoral dissertation. It isn't a protocol or a standard — it's a set of constraints. Anything that satisfies them is "RESTful"; most APIs described as REST satisfy only some.

The central idea: everything is a **resource**, addressed by a URI, manipulated through a small fixed set of HTTP methods.

---

## The Six Architectural Constraints

| Constraint | Meaning | Practical consequence |
|---|---|---|
| **Statelessness** | The server keeps no client session context. Every request carries everything needed to process it. | Auth token on every request. Any server instance can handle any request → horizontal scaling. |
| **Client–Server Separation** | UI and data storage evolve independently. | Same API serves web, mobile, and third parties. |
| **Uniform Interface** | Resources exposed via consistent URIs; actions uniform across the API. | A developer who learns one endpoint can guess the rest. |
| **Cacheability** | Responses declare themselves cacheable or not. | `Cache-Control`, `ETag` — fewer round trips. |
| **Layered System** | The client can't tell if it's talking to the origin server or an intermediary. | Load balancers, CDNs, API gateways, WAFs insert transparently. |
| **Code on Demand** *(optional)* | Servers may transfer executable code to extend the client. | Rarely used in practice. |

> **Statelessness ≠ no state.** Application state (users, orders) absolutely lives on the server, in a database. What's forbidden is *session* state held in server memory between requests.

### HATEOAS — the constraint everyone skips

Part of the Uniform Interface constraint: **Hypermedia As The Engine Of Application State**. Responses should include links telling the client what it can do next, so clients discover the API rather than hardcoding URLs.

```json
{
  "id": 123,
  "status": "pending",
  "_links": {
    "self":   { "href": "/v1/orders/123" },
    "cancel": { "href": "/v1/orders/123/cancel", "method": "POST" },
    "items":  { "href": "/v1/orders/123/items" }
  }
}
```

Almost no production API does this. Fielding's position is that an API without HATEOAS isn't really REST — which is why the honest label for most APIs is "HTTP JSON API."

### Richardson Maturity Model

A useful ladder for describing how RESTful something actually is:

- **Level 0** — Single endpoint, RPC over HTTP (SOAP-style: `POST /api` with an action in the body).
- **Level 1** — Multiple resources with distinct URIs, but still one method.
- **Level 2** — Proper HTTP methods and status codes. *Most real-world "REST" APIs stop here.*
- **Level 3** — HATEOAS / hypermedia controls.

---

## HTTP Methods & CRUD

Resources are exposed at URIs; the method determines the action.

| Method | CRUD | Purpose | Example |
|---|---|---|---|
| `GET` | Read | Retrieve a resource or collection | `GET /v1/users/123` |
| `POST` | Create | Create a new resource under a collection | `POST /v1/users` |
| `PUT` | Update | Replace a resource **entirely** | `PUT /v1/users/123` |
| `PATCH` | Update | Modify **part** of a resource | `PATCH /v1/users/123` |
| `DELETE` | Delete | Remove a resource | `DELETE /v1/users/123` |
| `HEAD` | — | Like GET, headers only (existence/size checks) | `HEAD /v1/users/123` |
| `OPTIONS` | — | Discover supported methods; CORS preflight | `OPTIONS /v1/users` |

### Safety & idempotency

Two properties that matter enormously for retries, caching, and proxy behaviour:

| Method | Safe (no side effects) | Idempotent (N calls = 1 call) |
|---|---|---|
| `GET` | Yes | Yes |
| `HEAD` | Yes | Yes |
| `OPTIONS` | Yes | Yes |
| `PUT` | No | **Yes** |
| `DELETE` | No | **Yes** |
| `POST` | No | **No** |
| `PATCH` | No | Not guaranteed |

**Why it matters:** a network timeout on `PUT` or `DELETE` can be safely retried. Retrying `POST` may create a duplicate record. Guard POST with a unique DB constraint or an `Idempotency-Key` header (the pattern Stripe uses).

### PUT vs PATCH — the practical difference

```http
PUT /v1/users/123
{ "name": "Harshit", "email": "h@example.com", "city": "Lucknow" }
→ Full replacement. Omitted fields get cleared or reset to defaults.

PATCH /v1/users/123
{ "city": "Lucknow" }
→ Only `city` changes. Everything else untouched.
```

Sending a partial payload to `PUT` is a common bug — it can silently wipe fields.

---

## Resource Naming

Resources are **nouns**; the method is the verb.

```
GET    /v1/products           ← collection (plural)
GET    /v1/products/42        ← single item
POST   /v1/products
GET    /v1/products/42/reviews    ← nested / sub-resource
```

Conventions:

- **Plural nouns** for collections — `/users`, not `/user`.
- **No verbs in paths** — `/v1/products`, not `/v1/getProducts`.
- **Lowercase, hyphen-separated** — `/purchase-orders`, not `/purchaseOrders` or `/purchase_orders`.
- **Limit nesting to one or two levels.** `/users/1/orders/5/items/9/reviews` is unusable — after the second level, expose the sub-resource at top level with a filter: `/reviews?itemId=9`.
- **No trailing slashes**, and no file extensions (`/users.json`) — use `Accept` headers instead.

### Actions that aren't CRUD

Some operations genuinely aren't resource manipulation — sending an email, cancelling an order, running a search. Options:

1. Model it as a sub-resource: `POST /v1/orders/123/cancellation`
2. Treat the action as a resource: `POST /v1/password-resets`
3. Pragmatic verb endpoint: `POST /v1/orders/123/cancel` — not pure REST, but clear. Most teams do this.

---

## Request Structure

| Component | Description |
|---|---|
| **Method** | The operation (`GET`, `POST`, …) |
| **URL / Endpoint** | `https://api.example.com/v1/users/123?fields=name,email` |
| **Path parameters** | Identify a specific resource — the `123` above |
| **Query parameters** | Modify the request — filtering, sorting, pagination, field selection |
| **Headers** | Metadata: auth, content negotiation, caching |
| **Body / Payload** | The representation being sent. Used with `POST`, `PUT`, `PATCH` |

### Headers worth knowing

| Header | Direction | Purpose |
|---|---|---|
| `Authorization: Bearer <token>` | → | Credentials |
| `Content-Type: application/json` | ↔ | Format of *this message's* body |
| `Accept: application/json` | → | Format the client *wants* back |
| `Accept-Language: en-IN` | → | Locale negotiation |
| `If-None-Match: "abc123"` | → | Conditional GET — return 304 if unchanged |
| `If-Match: "abc123"` | → | Optimistic concurrency — reject if changed |
| `ETag: "abc123"` | ← | Version fingerprint of the representation |
| `Cache-Control: max-age=300` | ← | Caching policy |
| `Location: /v1/users/123` | ← | URI of a newly created resource (with 201) |
| `Retry-After: 60` | ← | Backoff hint (with 429 or 503) |
| `X-RateLimit-Remaining: 42` | ← | Quota status |

---

## Response Structure

- **Status code** — three digits signalling outcome.
- **Headers** — caching, rate limits, `Location`, content type.
- **Body** — the representation, almost always JSON. (XML is legacy; JSON won because it's compact and maps directly to objects in most languages.)

---

## Status Codes

### 2xx — Success

| Code | Meaning |
|---|---|
| `200 OK` | Generic success with a body |
| `201 Created` | Resource created — include a `Location` header |
| `202 Accepted` | Accepted for async processing; not done yet |
| `204 No Content` | Success, deliberately empty body — typical for `DELETE` |

### 3xx — Redirection

| Code | Meaning |
|---|---|
| `301 / 308` | Permanent move (308 preserves the method) |
| `304 Not Modified` | Cached copy is still valid — no body sent |

### 4xx — Client error

| Code | Meaning |
|---|---|
| `400 Bad Request` | Malformed syntax or missing parameters |
| `401 Unauthorized` | Missing/invalid credentials — actually *unauthenticated* |
| `403 Forbidden` | Authenticated, but not permitted |
| `404 Not Found` | Resource or endpoint doesn't exist |
| `405 Method Not Allowed` | Wrong verb for this URI |
| `409 Conflict` | State conflict — duplicate email, version mismatch |
| `415 Unsupported Media Type` | Server can't parse the sent `Content-Type` |
| `422 Unprocessable Entity` | Syntactically valid, semantically wrong — failed validation |
| `429 Too Many Requests` | Rate limit exceeded — pair with `Retry-After` |

### 5xx — Server error

| Code | Meaning |
|---|---|
| `500 Internal Server Error` | Unhandled exception |
| `502 Bad Gateway` | Upstream returned garbage |
| `503 Service Unavailable` | Overloaded or in maintenance |
| `504 Gateway Timeout` | Upstream too slow |

**401 vs 403** is the most-confused pair: 401 means *we don't know who you are*; 403 means *we know, and you can't*.

---

## Error Response Format

Don't return bare strings. Return a consistent, machine-parseable shape across every endpoint. **RFC 9457 (Problem Details for HTTP APIs)** is the standard:

```json
{
  "type": "https://api.example.com/errors/validation-failed",
  "title": "Validation failed",
  "status": 422,
  "detail": "One or more fields are invalid.",
  "instance": "/v1/users",
  "errors": [
    { "field": "email", "message": "must be a valid email address" },
    { "field": "age",   "message": "must be a positive integer" }
  ]
}
```

Served with `Content-Type: application/problem+json`. Even if you don't adopt RFC 9457 verbatim, pick one shape and use it everywhere — clients shouldn't need per-endpoint error handling.

Never leak stack traces, SQL, or internal paths in a 500 response.

---

## Collection Query Patterns

Not part of the REST spec, but near-universal conventions:

```
# Pagination — offset/limit
GET /v1/products?page=2&limit=50

# Pagination — cursor (better for large, mutating datasets)
GET /v1/products?limit=50&cursor=eyJpZCI6MTIzfQ

# Filtering
GET /v1/products?category=electronics&status=active&price_lte=5000

# Sorting  (- prefix = descending)
GET /v1/products?sort=-created_at,name

# Sparse fieldsets — reduce payload size
GET /v1/products?fields=id,name,price

# Search
GET /v1/products?q=laptop

# Expanding relations — avoid N+1 client requests
GET /v1/orders/42?expand=customer,items
```

Return pagination metadata so the client knows where it is:

```json
{
  "data": [ /* ... */ ],
  "meta": { "page": 2, "limit": 50, "total": 1284, "total_pages": 26 },
  "links": {
    "next": "/v1/products?page=3&limit=50",
    "prev": "/v1/products?page=1&limit=50"
  }
}
```

**Offset vs cursor pagination:** offset is simple but drifts — if rows are inserted while a client pages through, items get skipped or duplicated, and deep offsets are slow in SQL. Cursor pagination is stable and fast but can't jump to an arbitrary page.

---

## Versioning

Breaking changes need a new version so existing integrations don't shatter.

| Strategy | Example | Notes |
|---|---|---|
| **URI path** | `/v1/users` | Most common, most visible, easiest to route and cache |
| Custom header | `X-API-Version: 2` | Cleaner URIs, harder to test and debug |
| `Accept` header | `Accept: application/vnd.example.v2+json` | Purist choice ("the URI identifies the resource, not its version") |
| Query param | `/users?version=2` | Easy, but messy alongside real filters |

**What counts as breaking:** removing or renaming a field, changing a type, adding a required request field, changing status-code semantics, tightening validation. **Non-breaking:** adding optional request fields, adding response fields, adding new endpoints. Design clients to tolerate unknown response fields and you'll need far fewer versions.

Also plan **deprecation**: announce it, send `Deprecation` / `Sunset` headers, give a migration window, then remove.

---

## Authentication & Authorization

| Mechanism | Use case |
|---|---|
| **API keys** | Server-to-server, simple identification. No user context. |
| **HTTP Basic** | Internal tools only. Credentials on every request — HTTPS mandatory. |
| **Bearer tokens / JWT** | Stateless auth. Self-contained and verifiable — but hard to revoke before expiry, so keep access tokens short-lived and pair with refresh tokens. |
| **OAuth 2.0 / OIDC** | Third-party delegated access ("Sign in with Google"). The standard for user-facing platforms. |
| **HMAC signatures** | Webhooks and high-integrity requests — sign the raw body. |
| **mTLS** | Internal service-to-service in zero-trust networks. |

Rules that aren't optional:

- **HTTPS everywhere.** No exceptions. Tokens in cleartext are compromised tokens.
- **Never put secrets in URLs** — they land in server logs, browser history, and `Referer` headers.
- **Authorize per resource,** not just per endpoint. `GET /v1/orders/123` must verify that order 123 belongs to the caller. Otherwise you have an IDOR vulnerability — one of the most common API flaws in the wild.
- **Rate limit** by key/user/IP to blunt abuse and credential stuffing.

---

## Caching

```http
Cache-Control: public, max-age=300, stale-while-revalidate=60
ETag: "33a64df551425fcc55e4d42a148795d9f25f89d4"
Last-Modified: Mon, 24 Aug 2026 09:00:00 GMT
```

Conditional-request flow:

1. Client sends `If-None-Match: "33a64df5..."`.
2. Unchanged → server returns **304 Not Modified** with no body.
3. Changed → server returns 200 with the new body and a new `ETag`.

`ETag` also enables **optimistic concurrency** on writes: send `If-Match: "<etag>"` with a `PUT`; the server returns **412 Precondition Failed** if someone else modified the resource meanwhile. This is how you avoid lost updates without locking.

Use `Cache-Control: no-store` for anything sensitive.

---

## Design Best Practices

- **Nouns, not verbs.** Let the method carry the action.
- **Version from day one.** Retrofitting `/v1/` later is painful.
- **Paginate every collection.** An unbounded list endpoint will eventually be your outage.
- **HTTPS only.**
- **Use the right status codes.** Returning `200 {"error": "not found"}` breaks every client, cache, and monitoring tool downstream.
- **Consistent casing.** Pick `camelCase` or `snake_case` for JSON keys and never mix. (`snake_case` is the more common API convention; `camelCase` is friendlier to JS clients.)
- **ISO 8601 UTC timestamps** — `2026-08-24T09:00:00Z`. Never local time, never epoch-without-units.
- **Validate at the boundary.** Zod/Joi/Pydantic on every incoming payload. Return 422 with field-level detail.
- **Never expose internal representations.** Map entities to DTOs so DB refactors don't break clients, and so you don't accidentally serialise a password hash.
- **Filter response fields explicitly** — allowlist, not blocklist.
- **Document with OpenAPI (Swagger).** A machine-readable spec gives you interactive docs, client SDK generation, request validation, and contract tests from one artefact.
- **Return the created/updated resource** in the response body — saves the client a follow-up GET.
- **Support idempotency keys** on non-idempotent writes if clients will retry (payments, orders).
- **Soft-delete carefully.** If `DELETE` only sets a flag, decide whether the resource then returns 404 or 410 — and be consistent.
- **Log with correlation IDs** (`X-Request-ID`) so a request can be traced across services.

---

## REST vs Alternatives

| | REST | GraphQL | gRPC |
|---|---|---|---|
| **Transport** | HTTP/1.1+ | HTTP (usually single POST endpoint) | HTTP/2 |
| **Payload** | JSON | JSON | Protobuf (binary) |
| **Shape of data** | Server decides | Client decides per query | Contract-defined |
| **Over/under-fetching** | Common problem | Solved by design | Contract-controlled |
| **Caching** | Native HTTP caching | Hard — needs custom layer | Hard |
| **Discoverability** | OpenAPI | Introspection + strong typing | `.proto` files |
| **Best for** | Public APIs, CRUD, broad compatibility | Complex nested reads, many clients with different needs | Low-latency internal microservices, streaming |

REST remains the default for public-facing APIs because HTTP semantics, caching, tooling, and debuggability come free. GraphQL earns its complexity when clients need wildly different slices of a deeply nested graph. gRPC wins inside the datacentre where latency and schema enforcement matter more than curl-ability.

---

## Testing & Tooling

- **curl** / **HTTPie** — quick manual calls
- **Postman** / **Insomnia** / **Bruno** — collections, environments, saved auth
- **Thunder Client** / **REST Client** — VS Code extensions, `.http` files in the repo
- **Supertest + Jest/Vitest** — integration tests for Express apps
- **Swagger UI / Redoc** — rendered docs from an OpenAPI spec
- **k6** / **Artillery** — load testing

```bash
curl -X POST https://api.example.com/v1/users \
  -H "Content-Type: application/json" \
  -H "Authorization: Bearer $TOKEN" \
  -d '{"name":"Harshit","email":"h@example.com"}' \
  -i    # include response headers — where the interesting information usually is
```

---

## Related

- [[Express POST Routes]]
- [[Parsing Request Bodies in JavaScript]]
- [[HTTP Status Codes]]
- [[Authentication & JWT]]
- [[OpenAPI / Swagger]]

## References

- [Fielding, *Architectural Styles and the Design of Network-based Software Architectures* (2000), Ch. 5](https://ics.uci.edu/~fielding/pubs/dissertation/rest_arch_style.htm)
- [MDN — HTTP methods](https://developer.mozilla.org/en-US/docs/Web/HTTP/Methods)
- [MDN — HTTP status codes](https://developer.mozilla.org/en-US/docs/Web/HTTP/Status)
- [RFC 9110 — HTTP Semantics](https://www.rfc-editor.org/rfc/rfc9110.html)
- [RFC 9457 — Problem Details for HTTP APIs](https://www.rfc-editor.org/rfc/rfc9457.html)
- [Microsoft REST API Guidelines](https://github.com/microsoft/api-guidelines)
- [OpenAPI Specification](https://spec.openapis.org/oas/latest.html)
